# 04 — ComBat harmonization: does correcting the site confound at the feature level beat the plain `combined` baseline?

**Decision this feeds (`structuring-ml-projects` step 6, the gate):** the
classical baseline (`abs_asym` + `striatal_ratio`, `notebooks/03_baseline_classical.ipynb`)
already cleared its own gate at `combined = 0.5753` log loss. EDA section
3a found a real, significant target-rate confound (chi2=24.54, dof=6,
p=0.00042) tied to **in-plane spacing family** — the same variable
`evaluate.make_folds` already stratifies folds on. ComBat (Fortin et al.
2018, `RESOURCES.md`) is the standard tool for removing a site effect from
*derived scalar features* (as opposed to per-volume intensity
normalization, which only fixes scale, not a feature-level shift — see
the Frontiers 2022 PPMI radiomics precedent also logged in
`RESOURCES.md`, AUC 0.71→0.77 after ComBat-GAM on the same kind of
confound).

If `combined + ComBat` doesn't beat plain `combined` by more than the
noise floor (this run's own per-variant sd), harmonization is a negative
result (SKILL.md step 7) — log it, don't wire it in.

**Method (`src/features.py::fit_combat`/`apply_combat`, TDD'd against
synthetic batch-shifted data, `tests/test_features.py`):** parametric
empirical-Bayes ComBat (Johnson et al. 2007), batch = in-plane spacing
family. **Fit-on-controls**, per the Frontiers 2022 leakage-avoidance
detail: within each CV fold, harmonization parameters are estimated from
that fold's **training-set normal (label=0) volumes only**, then applied
to every row (both classes, train and test) via the same fitted
per-batch parameters — the batch identity (spacing family) is a known
physical property at inference time, not something derived from the
label, so this doesn't leak.

**Data handling:** the cell below only reads
`data/processed/baseline_features.csv` (derived scalar features + label,
already written by `03`, no new `.nii.gz` access), but per the AI-assistant
data rule (`README.md`) it still touches row-level labels, so it stays
**[RUN ME]** — run it yourself, share back only the printed aggregate
comparison table.

In [ ]:
# [RUN ME] -- reads data/processed/baseline_features.csv (no .nii.gz access),
# but touches row-level labels, so per the AI-assistant data rule this is
# for you to run -- share back only the printed comparison table.
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path.cwd().parent / "src"))
import config
import evaluate
import features

feat_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")
y = feat_df[config.TARGET_COLUMN].to_numpy()
family = feat_df["inplane_family"].to_numpy()
X_all = feat_df[["abs_asym", "striatal_ratio"]].to_numpy(dtype=float)

N_SEEDS = 5


def run(use_combat):
    """Same fold structure/model as 03_baseline_classical.ipynb's `combined`
    variant; ComBat (fit on that fold's train-set controls) is inserted
    right before StandardScaler when use_combat=True."""
    seed_scores = []
    for seed in range(N_SEEDS):
        folds = evaluate.make_folds(y, family, n_splits=config.N_FOLDS, random_state=seed)
        preds = np.zeros(len(y))
        for train_idx, test_idx in folds:
            X_train, X_test = X_all[train_idx].copy(), X_all[test_idx].copy()
            if use_combat:
                controls = train_idx[y[train_idx] == 0]
                params = features.fit_combat(X_all[controls], family[controls])
                X_train = features.apply_combat(X_train, family[train_idx], params)
                X_test = features.apply_combat(X_test, family[test_idx], params)
            model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
            model.fit(X_train, y[train_idx])
            preds[test_idx] = model.predict_proba(X_test)[:, 1]
        seed_scores.append(evaluate.log_loss_score(y, preds))
    return np.array(seed_scores)


# Baseline (`combined`, no ComBat) is recomputed in this same run -- cheap,
# and this notebook's fold loop is otherwise identical to 03's, so this is
# the correct noise-floor reference for the ComBat delta below (03's own
# sd used a slightly different code path -- eda_features.csv era -- so
# don't reuse it across notebooks; SKILL.md step 4).
baseline_scores = run(use_combat=False)
combat_scores = run(use_combat=True)

print(f"{'variant':<22} {'mean':>8} {'sd':>8} {'vs baseline':>14}")
for name, scores in [("combined (no ComBat)", baseline_scores),
                      ("combined + ComBat", combat_scores)]:
    print(f"{name:<22} {scores.mean():>8.4f} {scores.std():>8.4f} "
          f"{scores.mean() - config.BASELINE_LOGLOSS:>+14.4f}")

delta = combat_scores.mean() - baseline_scores.mean()
print()
print(f"ComBat vs. no-ComBat (combined features): delta = {delta:+.4f}")


**What we're looking for:** whether removing the in-plane-spacing-family
site effect from `abs_asym`/`striatal_ratio` via ComBat improves the
classical baseline beyond noise, or whether the model already handles the
confound well enough on its own (via fold stratification on the same
variable) that harmonization adds nothing.

**Why:** `SKILL.md` step 6 — only wire in what wins, and step 3 — this is
the standard tool for a feature-level site effect (Fortin et al. 2018),
with a direct precedent on the same kind of confound and modality
(Frontiers 2022 PPMI radiomics study, `RESOURCES.md`).

**Source:** Fortin et al. 2018 (NeuroImage 167:104-120); Frontiers in
Neuroscience 2022 PPMI multicenter harmonization study (leakage-avoidance
precedent for fit-on-controls). Both in `RESOURCES.md`.

**What we found** (run 2026-09-08, 5 seeds x `config.N_FOLDS`,
`abs_asym` + `striatal_ratio`, batch = in-plane spacing family, fit on
each fold's training-set controls only):

| variant | mean | sd | vs. baseline |
|---|---|---|---|
| combined (no ComBat) | 0.5753 | 0.0006 | -0.1132 |
| **combined + ComBat** | **0.5290** | **0.0011** | **-0.1594** |

ComBat vs. no-ComBat: delta = **-0.0462**. The noise floor (per-variant
seed sd) is 0.0006-0.0011; the delta is ~40-80x that — far outside noise,
not a coin flip. This confirms the Frontiers 2022 precedent's shape of
result (ComBat improving a feature-level site confound beyond what
per-volume normalization and fold stratification alone achieve) on this
dataset's own confound (in-plane spacing family, χ²=24.54, p=0.00042,
EDA section 3a).

**Decision / next step:** the gate is **passed** — `combined + ComBat`
beats plain `combined` by far more than this run's own sd.
`fit_combat`/`apply_combat` are wired into `src/model.py` as
`ComBatHarmonizedPipeline` / `build_combat_baseline()` (TDD,
`tests/test_model.py`), fit on training-set controls exactly as here.
`build_combat_baseline()` is now the classical baseline to actually use;
`build_classical_baseline()` (no ComBat) stays as the weaker reference
point it was validated against.